In [ ]:
from pyspark.sql.functions import col

from olist_silver.transformations import (
    is_valid_uuid,
    merge_into,
    null_invalid_uuid,
    parse_timestamp,
    with_processed_timestamp,
)

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_order_items_table_name = dbutils.widgets.get("raw_olist_order_items_table")

silver_schema = dbutils.widgets.get("silver_schema")
order_items_table_name = dbutils.widgets.get("order_items_table")

In [ ]:
raw_olist_order_items_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_order_items_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{order_items_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{order_items_table_name} (
            orderId STRING,
            orderItemId INT,
            productId STRING,
            sellerId STRING,
            shippingLimitDate TIMESTAMP,
            price DOUBLE,
            freightValue DOUBLE,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
order_items_silver_df = with_processed_timestamp(
    raw_olist_order_items_df.where(is_valid_uuid("order_id") & col("order_item_id").cast("int").isNotNull())
    .select(
        col("order_id").cast("string").alias("orderId"),
        col("order_item_id").cast("int").alias("orderItemId"),
        null_invalid_uuid("product_id").cast("string").alias("productId"),
        null_invalid_uuid("seller_id").cast("string").alias("sellerId"),
        parse_timestamp("shipping_limit_date").alias("shippingLimitDate"),
        col("price").cast("double").alias("price"),
        col("freight_value").cast("double").alias("freightValue"),
    )
    .dropDuplicates(["orderId", "orderItemId"])
)

In [ ]:
merge_into(
    spark,
    target=f"{catalog}.{silver_schema}.{order_items_table_name}",
    source_view="order_items_silver_view",
    keys=["orderId", "orderItemId"],
    source_df=order_items_silver_df,
)